In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Setup

We've seen already that any first-order ODE can be described by
\begin{align}
    \frac{dy}{dx} &= f(x, y).
\end{align}

In [ ]:
class FirstOrderODE:
    '''
    A first-order ODE is an equation that specifies the slope of a function y(x)
    in terms of
     - the value of the function itself, y
     - the value of the variable it depends on, x.
    
    The generic form for an ODE is
    
    dy
    --  = f(x, y)
    dx
    
    '''

    def f(self, x, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f.')

    def exact_solution(self, x, *args):
        raise NotImplementedError('The exact solution is not known (or at least not provided).')
    
    def plot_slope_field(self, ax, x, y, color='black', zorder=-1, **kwargs):
        X, Y = np.meshgrid(x, y)

        # tan(theta) = slope
        # theta = arctan(slope)
        THETA = np.arctan(self.f(X, Y))
        U = np.cos(THETA)
        V = np.sin(THETA)
    
        ax.quiver(X, Y, U, V, color=color, angles='xy', pivot='middle', zorder=zorder, **kwargs)

and *integrated* to find (or numerically estimate) $y(x)$.

In [ ]:
class FirstOrderODEIntegrator:
    '''
    An integrator is an algorithm for producing numerical solutions to ODEs.
    '''

    def __init__(self, ODE):

        self.ODE = ODE

    def starting_from(self, x_0, y_0):
        raise NotImplementedError('No integration algorithm has been specified.')


*Euler's method* was the first integrator we described.  We found it by using the Taylor expansion truncated at first order,
\begin{align}
    y(x + \Delta x) &= y(x) + y'(x) \Delta x + \mathcal{O}(\Delta x^2)
\end{align}
and saw that it had *global error* (the error accumulated after many steps) that scaled like $\Delta x$.  Each step has an error that's only $\Delta x^2$ but over a fixed interval the number of steps we take scales like $1/\Delta x$.  So the total error is the product $\Delta x^2 \times 1/\Delta x = \Delta x^1$.

We saw that this means our numerical errors halve if we halve $\Delta x$.

In [ ]:
class EulerMethod(FirstOrderODEIntegrator):
    '''
    Euler's method is based on the Taylor series approximation

    y(x + Δx) = y(x) + y'(x) Δx + (higher-order terms with more powers of Δx which we neglect)

    Given a starting location on the solution (x_0, y_0) we can repeatedly say
    - the next x will be x_0 + Δx
    - the next y will be y_0 + Δy, and using the Taylor series Δy = y'(x) Δx
    
    This produces the next point on the solution, and we can begin again.
    '''

    def __init__(self, ODE, Delta_x, steps):

        self.Delta_x = Delta_x
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, x_0, y_0):

        # Populate all the x values
        x = x_0 + self.Delta_x * np.arange(self.steps+1)
        # and allocate an array to store the y values we compute
        y = np.zeros_like(x, dtype=float)

        # The user picked a starting y value
        y[0] = y_0

        # and we can step through the Euler procedure over and over again.
        for step in range(self.steps):
            slope = self.ODE.f(x[step], y[step])
            y[step+1] = y[step] + slope * self.Delta_x

        return x, y

# Second-Order Methods

In object-oriented-odes.ipynb we developed the idea of a *higher-order method* for integrating a first-order ODE.  There we implemented RK2 but we first understood the more direct generalization of the Euler method,
\begin{align}
    \Delta y &\approx y'(x) \Delta x + \frac{1}{2} y''(x) \Delta x^2 + \mathcal{O}(\Delta x^3)
    \\
    &= f(x, y) \Delta x + \frac{1}{2} \frac{df}{dx} \Delta x^2 + \mathcal{O}(\Delta x^3)
    \\
    &= f(x, y) \Delta x + \frac{1}{2} \left(\partial_x f + \partial_y f \times \frac{dy}{dx}\right)\Delta x^2 + \mathcal{O}(\Delta x^3)
    \\
    &= f(x, y) \Delta x + \frac{1}{2} \left(\partial_x f(x, y) + \partial_y f(x, y) \times f(x, y)\right) \Delta x^2 + \mathcal{O}(\Delta x^3)
\end{align}
This is called the *second-order Taylor method*.  It's second-order because the mistake we make in each step is of size $\Delta x^3$ and so the global error accumulated over lots of steps each of size $\Delta x$ is $\Delta x^2$.

We found $y''$ by differentiating the original differential equation $y''(x) = \frac{d}{dx} f(x, y)$ and the second to last line is because $y$ implicitly depends on $x$.  Finally, we used the differential equation to replace $dy/dx = f(x, y)$.

One of the issues with this method is that our `FirstOrderODE` class didn't support differentiating the `f` method,

But, we can subclass the `FirstOrderODE` so that it supplies the needed methods.

In [ ]:
class DifferentiableFirstOrderODE(FirstOrderODE):

    def partial_derivatives_of_f(self, x, y):
        raise NotImplementedError('The generic ODE has an unspecified slope function f and therefore unspecified partial derivatives.')
        # return a tuple (∂_x f, ∂_y f)
    
    def dfdx(self, x, y):
        # This method is *generic*.  So long as the partial derivatives are provided we can compute the total derivative using
        #
        # df/dx = ∂_x f + (∂_y f) * dy/dx
        #
        # and we know dy/dx = f for any FirstOrderODE.
        partial_x, partial_y = self.partial_derivatives_of_f(x, y)
        return partial_x + partial_y * self.f(x, y)

so that if someone provides the needed partial derivatives we can compute $df/dx$.

Let's promote our logistic ODE, given by $f(x, y) = \frac{k}{L} y \times (L-y)$,

In [ ]:
class LogisticODE(FirstOrderODE):
    '''
    The logistic ODE is a first-order ODE which has f(x, y) = k/L y (L - y).
    '''

    def __init__(self, k, L):
        self.k = k
        self.L = L

    def f(self, x, y):
        return self.k / self.L * y * (self.L-y)

    def exact_solution(self, x, x_0=0):
        return self.L / (1 + np.exp(-self.k * (x-x_0)))

to a fully `DifferentiableFirstOrderODE` named `DifferentiableLogisticODE`.

In [ ]:
class DifferentiableLogisticODE(DifferentiableFirstOrderODE):
    '''
    The logistic ODE is a first-order ODE which has f(x, y) = k/L y (L - y).
    '''

    def __init__(self, k, L):
        self.k = k
        self.L = L

    def f(self, x, y):
        return self.k / self.L * y * (self.L-y)

    def partial_derivatives_of_f(self, x, y):
        raise NotImplementedError('Please make partial_derivatives_of_f return the tuple (∂_x f, ∂_y f)')
        partial_x = ...
        partial_y = ...
        return (partial_x, partial_y)
    
    def exact_solution(self, x, x_0=0):
        return self.L / (1 + np.exp(-self.k * (x-x_0)))

Now let's implement the second-order Taylor method,
\begin{align}
    \Delta y &\approx y'(x) \Delta x + \frac{1}{2} y''(x) \Delta x^2 + \mathcal{O}(\Delta x^3)
    \\
    &= f(x, y) \Delta x + \frac{1}{2} \frac{df}{dx} \Delta x^2 + \mathcal{O}(\Delta x^3)
\end{align}

In [ ]:
class SecondOrderTaylorMethod(FirstOrderODEIntegrator):

    def __init__(self, ODE, Delta_x, steps):

        self.Delta_x = Delta_x
        self.steps = steps

        super().__init__(ODE)

    def starting_from(self, x_0, y_0):

        # Populate all the x values
        x = x_0 + self.Delta_x * np.arange(self.steps+1)
        # and allocate an array to store the y values we compute
        y = np.zeros_like(x, dtype=float)

        # The user picked a starting y value
        y[0] = y_0

        raise NotImplementedError('Please implement the second-order Taylor method.')
        return x, y

Now let's compare how the errors scale with $\Delta x$.

In [ ]:
k, L = 1., 2.
logistic = DifferentiableLogisticODE(k, L)

DELTA_X = 1.0/2**np.arange(0, 15)

EULER_AT_END = np.zeros_like(DELTA_X)
TAYLOR2_AT_END = np.zeros_like(DELTA_X)

fig, ax = plt.subplots()
logistic.plot_slope_field(ax, np.linspace(0, 1, 20), np.linspace(1, 1.6, 20), alpha=0.25)

for i, Delta_x in enumerate(DELTA_X):
    integrator = EulerMethod(logistic, Delta_x, int(1/Delta_x))
    x, y = integrator.starting_from(0, 1)
    EULER_AT_END[i] = y[-1]
    if Delta_x >= 0.25:
        ax.plot(x, y, label=f'Euler Δx={Delta_x}')
    
    integrator = SecondOrderTaylorMethod(logistic, Delta_x, int(1/Delta_x))
    x, y = integrator.starting_from(0, 1)
    TAYLOR2_AT_END[i] = y[-1]
    if Delta_x >= 0.25:
        ax.plot(x, y, label=f'Taylor2 Δx={Delta_x}')
        pass

ax.plot(x, logistic.exact_solution(x), label='exact solution', color='black')
ax.legend()

exact_value = logistic.exact_solution(1)
euler_error = np.abs(EULER_AT_END - exact_value)
taylor2_error = np.abs(TAYLOR2_AT_END - exact_value)

In [ ]:
fig, axes = plt.subplots(1,2, figsize=(16,5))
fig.suptitle('Error after integrating to x=1 from (x_0, y_0) = (0, 1)')

for axis in axes:
    axis.plot(DELTA_X, euler_error, marker='o', linestyle='none', label='Euler')
    axis.plot(DELTA_X, taylor2_error, marker='o', linestyle='none', label=f'Taylor 2')
    axis.set_xlabel('Δx')
    axis.set_ylabel('Error = |numerical - exact|')

axes[0].legend()
axes[1].set_xscale('log')
axes[1].set_yscale('log')

**Describe what you see.  Which method is more accurate at the same $\Delta x$?  How do the errors scale differently?** 

# A New ODE With No Known Exact Solution

The logistic equation can be used to model population dynamics, where $L$ is the equilibrium population.  But some species have a seasonal boom and bust cycle, which we can model by allowing $L$ to depend on ~time~ $x$ in an oscillatory manner.

This can be modeled by an ODE you have not met before, given by
\begin{align}
    \frac{dy}{dx} &= \frac{k}{L(x)} y(x) (L(x) - y(x))
    &
    L(x) &= L_0 \times (1 + \epsilon \sin \omega x)
\end{align}

There is no known exact solution to this equation.  To get the numerical error you can take the finest integration and use it as 'the truth' (even though it is not!) against which to compare other integrations.

In [ ]:
class OscillatoryODE(DifferentiableFirstOrderODE):

    def __init__(self, k, L0, epsilon, omega):

        raise NotImplementedError('Please implement the logistic differential equation with seasonal booms and busts described above.')

    def L(self, x):
        raise NotImplementedError('Implement L(x)')

    def dLdx(self, x):
        raise NotImplementedError('You will find that you need dL/dx to compute the ∂_x f')
        
    ... # also implement f, partial_derivatives_of_f

Please compare the performance of the Euler method and the second-order Taylor method for $(k, L_0, \epsilon, \omega) = (1., 2., 0.75, 4)$ integrated from $x=0$ to $x=4$ with a variety of
 - step sizes $\Delta x$, and
 - initial $y_0$ values $(0.5, 1, 2, 4)$.

Plot your numerical solutions as a function of $x$.

In [ ]:
k, L0 = 1., 2.
epsilon, omega = 0.75, 4
oscillatory = OscillatoryODE(k, L0, epsilon, omega)

x_0 = 0
end = 4
Y_0 = (0.5, 1, 2, 4)

raise NotImplementedError('''
For each initial y_0 integrate with both Euler and second-order Taylor
and plot the solution as a function of x.
''')


Take your best numerical solution for each $y_0$ as your best estimate of the truth and call the error at any $\Delta x$ the absolute difference with that 'truth'.  Plot the error as a function of $\Delta x$  for each $y_0$.

In [ ]:
raise NotImplementedError('Please plot the scaling of the error as a function of Δx')

**Describe what you see.  Which method is more accurate?  How do the errors scale?**